# Phase 5 Orientation — From Scripts to Projects
**Project Prometheus**

---

## What This Is

Phase 5 was designed as a standalone phase on professional software engineering practices — tests, Docker, CI/CD, packaging.

**We're covering it as an orientation**, not a full phase. Here's why:

- **Git discipline**: already covered in your reference guide. You're actively using it.
- **pytest basics**: you'll learn it properly when you write tests for your milestone project — learning-by-doing is more effective than a standalone unit.
- **Docker**: worth knowing exists and understanding conceptually. Not interview-critical by Feb 2027, but useful awareness.
- **CI/CD**: 30 minutes to set up GitHub Actions when you need it. Not worth a full phase now.

> 💡 **The milestone project teaches Phase 5 skills more effectively than a dedicated phase.** Building it forces real test-writing, real Git discipline, and real reproducibility decisions.
---


---
## 1️⃣ pytest — The Basics You Actually Need

You need enough pytest to write tests for your milestone project. That's about 20% of pytest's features.


In [ ]:
# How to write tests — the essentials
# File must be named test_*.py or *_test.py
# Functions must start with test_
# pytest discovers and runs them automatically

import numpy as np
import pandas as pd

# --- tests/test_features.py (example) ---

def annualise_vol(daily_vol: float, trading_days: int = 252) -> float:
    # Annualise a daily volatility figure
    return daily_vol * np.sqrt(trading_days)

def compute_rolling_vol(returns: pd.Series, window: int = 20) -> pd.Series:
    # Compute rolling annualised volatility
    return returns.rolling(window).std() * np.sqrt(252)

# --- The actual tests ---

def test_annualise_vol_basic():
    # Basic sanity check: 1% daily vol -> ~15.87% annual
    result = annualise_vol(0.01)
    expected = 0.01 * np.sqrt(252)
    assert abs(result - expected) < 1e-10, f"Expected {expected}, got {result}"

def test_annualise_vol_zero():
    # Edge case: zero vol stays zero
    assert annualise_vol(0.0) == 0.0

def test_annualise_vol_custom_days():
    # Custom trading days
    result = annualise_vol(0.01, trading_days=365)
    expected = 0.01 * np.sqrt(365)
    assert abs(result - expected) < 1e-10

def test_rolling_vol_output_shape():
    # Rolling vol should have same length as input
    returns = pd.Series(np.random.normal(0, 0.01, 100))
    vol = compute_rolling_vol(returns, window=20)
    assert len(vol) == len(returns), "Output length should match input length"

def test_rolling_vol_nan_at_start():
    # First (window-1) values should be NaN
    returns = pd.Series(np.random.normal(0, 0.01, 100))
    vol = compute_rolling_vol(returns, window=20)
    assert vol.iloc[:19].isna().all(), "First 19 values should be NaN for window=20"
    assert not np.isnan(vol.iloc[19]), "Value at index 19 should not be NaN"

# Run them inline (normally you'd use: pytest tests/ -v)
import traceback

tests = [
    test_annualise_vol_basic,
    test_annualise_vol_zero,
    test_annualise_vol_custom_days,
    test_rolling_vol_output_shape,
    test_rolling_vol_nan_at_start,
]

for test_fn in tests:
    try:
        test_fn()
        print(f"  ✅ {test_fn.__name__}")
    except AssertionError as e:
        print(f"  ❌ {test_fn.__name__}: {e}")
    except Exception as e:
        print(f"  ❌ {test_fn.__name__}: {traceback.format_exc()}")


---
## 2️⃣ Test Categories — What to Test

| Test Type | What it covers | Example |
|-----------|---------------|---------|
| Unit test | One function in isolation | `test_annualise_vol_basic()` |
| Integration test | Multiple components together | `test_full_pipeline_runs()` |
| Regression test | Catches bugs introduced by changes | Lock output hash, fail if it changes |
| Smoke test | Does it even start? | `test_pipeline_imports()` |

**Testing priorities for your milestone project:**

1. ✅ Corporate action adjustments — test against known values (AAPL 4-for-1)
2. ✅ Variance scaling — test the annualisation formula
3. ✅ No NaN leakage — test that ffill doesn't propagate beyond threshold
4. ✅ Pipeline determinism — test same input → same hash
5. ✅ Validation catches known bad data — test with deliberately broken input

**What NOT to unit test:**
- yfinance download itself (test your wrapper logic, mock the API call)
- Plotting functions (visual regression testing is overkill here)
- Constants and config values


---
## 3️⃣ Docker — Conceptual Understanding

Docker solves the "works on my machine" problem.

**Without Docker:**
- Your pipeline runs on Python 3.11, pandas 2.2, MacOS ARM
- Someone else runs on Python 3.9, pandas 1.5, Ubuntu x86
- Results differ. Debugging takes hours.

**With Docker:**
- You package your entire environment into an image
- Anyone with Docker can run `docker run your-pipeline` and get identical results
- The container includes: Python version, all libraries, your code

**The Dockerfile (simplified):**
```dockerfile
FROM python:3.11-slim         # Base image: Python 3.11 on minimal Linux

WORKDIR /app                  # Working directory inside container

COPY requirements.txt .       # Copy dependency list
RUN pip install -r requirements.txt  # Install exact versions

COPY src/ ./src/              # Copy your code
COPY config/ ./config/

CMD ["python", "src/pipeline.py"]  # Default command
```

**Basic commands:**
```bash
docker build -t my-pipeline .           # Build image from Dockerfile
docker run my-pipeline                  # Run container
docker run -v $(pwd)/data:/app/data my-pipeline  # Mount local data folder
```

> 💡 **For your milestone project timeline:** Don't dockerise until the pipeline is working. Add Docker as the final step — it's a 1-2 hour addition once your code is stable.


---
## 4️⃣ GitHub Actions CI/CD — Conceptual Understanding

CI/CD (Continuous Integration / Continuous Deployment) automatically runs your tests every time you push code.

**What it does:**
1. You push code to GitHub
2. GitHub Actions triggers automatically
3. It spins up a clean Linux environment
4. Installs your requirements
5. Runs `pytest tests/`
6. Reports pass/fail on your pull request

**A minimal `.github/workflows/tests.yml`:**
```yaml
name: Run Tests

on: [push, pull_request]

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - name: Set up Python
        uses: actions/setup-python@v4
        with:
          python-version: '3.11'
      - name: Install dependencies
        run: pip install -r requirements.txt
      - name: Run tests
        run: pytest tests/ -v
```

That's it. Commit this file to `.github/workflows/tests.yml` and GitHub runs your tests on every push.

> 💡 **Why it matters for your portfolio:** A green CI badge on your GitHub repo signals professional practice. It means your code demonstrably works, not just "it worked when I wrote it."


---
## 5️⃣ Milestone Project Prep — Checklist

The milestone project is your Phase 3+4 capstone. Before you start building:

### What It Needs to Do
- [ ] Ingest daily prices for a configurable universe (yfinance + retry logic)
- [ ] Validate raw data (catch the common errors)
- [ ] Handle corporate actions correctly (validated against known split dates)
- [ ] Clean missing data (documented decisions per data type)
- [ ] Compute base features (returns, rolling vol, correlations)
- [ ] Persist to Parquet with metadata
- [ ] Run deterministically (same input → same hash)
- [ ] Have unit tests for each transformation step (>85% coverage)
- [ ] Have a README that lets someone else run it without asking you

### Git Structure to Follow
```
milestone-data-pipeline/
├── README.md
├── requirements.txt
├── config/
│   └── tickers.yaml
├── src/
│   ├── fetcher.py
│   ├── validator.py
│   ├── cleaner.py
│   ├── features.py
│   └── pipeline.py
├── tests/
│   ├── test_validator.py
│   ├── test_cleaner.py
│   ├── test_features.py
│   └── test_integration.py
└── .github/
    └── workflows/
        └── tests.yml
```

### Your Exit Condition
> You can run `git clone [your-repo]`, `pip install -r requirements.txt`, `python src/pipeline.py` — and get identical results to when you built it, on any machine.

---

## Phase 3+4+5 Combined Exit Criteria ✅

- [ ] Can explain the difference between statistical and economic significance
- [ ] Can spot data leakage in someone else's code in under 2 minutes
- [ ] Can describe walk-forward CV with purging and embargo from memory
- [ ] Know when to use Parquet vs CSV vs SQL
- [ ] Have written at least 10 meaningful unit tests
- [ ] Pipeline runs deterministically (verified by hash comparison)
- [ ] Can explain every design decision in your milestone project under interview conditions
